# Bài 6.4 · Language model: đoán từ tiếp theo

**Câu hỏi của buổi này:** một language model chỉ làm một việc là cho xác suất của token tiếp theo. Từ việc ấy, làm sao ra được loss, perplexity, và những đoạn văn sinh ra rất khác nhau tùy cách chọn token?

| Mục | Sẽ thấy |
|---|---|
| 1. Bigram tự đếm | bảng đếm cặp từ trên WikiText-2, câu bigram sinh ra |
| 2. GPT-2 đoán token tiếp theo | biểu đồ 10 token xác suất cao nhất |
| 3. Loss của từng token | câu tô màu theo độ bất ngờ, label dời một vị trí |
| 4. Cách chọn token | greedy, temperature, top-k, top-p: văn bản và số đo |
| 5. Greedy không tối ưu cả chuỗi | log-xác suất cả chuỗi của greedy và beam search |
| 6. Perplexity của người và máy | văn bản người viết, máy sinh, bị xáo trộn |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 6.4](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/6.4).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import math, re, collections, html
import torch, transformers, datasets
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
transformers.logging.set_verbosity_error(); datasets.disable_progress_bars()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
with quiet():
    tok = GPT2TokenizerFast.from_pretrained('gpt2'); lm = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()
    wiki = datasets.load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__), ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'),
                    ('GPT-2', f'{sum(p.numel() for p in lm.parameters()) / 1e6:.0f} triệu tham số, từ vựng {len(tok):,} token')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Bigram tự đếm

Model đơn giản nhất: xác suất từ tiếp theo chỉ phụ thuộc **một** từ ngay trước, ước lượng bằng cách đếm: $P(w_t \mid w_{t-1}) = \dfrac{\text{đếm}(w_{t-1}, w_t)}{\text{đếm}(w_{t-1})}$. Đếm trên phần train của WikiText-2 (khoảng 2 triệu từ).

In [ ]:
words = re.findall(r"[a-z]+|[.,]", ' '.join(wiki['train']['text']).lower())
uni = collections.Counter(words); bi = collections.Counter(zip(words, words[1:]))
after_the = pd.DataFrame([(b, n, n / uni['the']) for (a, b), n in bi.most_common() if a == 'the'][:8], columns=['từ sau "the"', 'số lần', 'P(từ | the)'])
table(after_the, f'{len(words):,} từ, {len(uni):,} từ khác nhau; tám từ hay đứng sau "the" nhất', fmt={'số lần': '{:,}', 'P(từ | the)': '{:.3f}'})

nxt = collections.defaultdict(list)
for (a, b), n in bi.items(): nxt[a].append((b, n))
def bigram_sample(start, n=25, seed=0):
    r = np.random.default_rng(seed); out = [start]
    for _ in range(n):
        cand = nxt[out[-1]]; w, c = zip(*cand); out.append(r.choice(w, p=np.array(c) / sum(c)))
    return ' '.join(out)
table(pd.DataFrame({'câu bigram sinh ra': [bigram_sample('the', seed=s) for s in range(3)]}), 'Bắt đầu bằng "the", mỗi bước chỉ nhìn một từ trước')
check('xác suất sau "the" cộng lại bằng 1', abs(sum(n for (a, b), n in bi.items() if a == 'the') / uni['the'] - 1) < 0.01)
note('Từng cặp từ liền nhau nghe ổn, nhưng cả câu không có nghĩa: model không nhớ gì xa hơn một từ.')

## 2. GPT-2 đoán token tiếp theo

GPT-2 nhìn **toàn bộ** phần đã viết. Đầu ra ở mỗi vị trí là 50 257 logits; softmax cho phân phối xác suất của token tiếp theo.

In [ ]:
prompts = ['The capital of France is', 'Vietnam is a country in', 'The cat sat on the']
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
rows = []
for ax, pr in zip(axes, prompts):
    ids = tok(pr, return_tensors='pt').input_ids.to(device)
    with torch.no_grad(): p = lm(ids).logits[0, -1].softmax(-1)
    top = p.topk(10)
    labels = [tok.decode(int(i)).replace(' ', '·') for i in top.indices]
    ax.barh(range(10)[::-1], top.values.cpu(), color=[C['blue']] + [C['gray']] * 9); ax.set_yticks(range(10)[::-1], labels)
    ax.set_title(f'"{pr} …"', fontsize=11); ax.grid(axis='y', visible=False); bar_labels(ax, '{:.2f}')
    rows.append((pr, labels[0], top.values[0].item(), top.values.sum().item()))
plt.tight_layout(); plt.show()
table(pd.DataFrame(rows, columns=['ngữ cảnh', 'token cao nhất', 'xác suất', 'tổng 10 token đầu']), 'Phân phối token tiếp theo (· là dấu cách)', fmt={'xác suất': '{:.3f}', 'tổng 10 token đầu': '{:.3f}'})
top_pred = rows[0]
with torch.no_grad():
    p_paris = lm(tok(prompts[0], return_tensors='pt').input_ids.to(device)).logits[0, -1].softmax(-1)[tok(' Paris').input_ids[0]].item()
note(f'Sau "{prompts[0]}", token đứng đầu là "{rows[0][1]}" ({rows[0][2]:.0%}), còn "·Paris" chỉ được {p_paris:.0%}: model đoán <b>token tiếp theo hợp lý</b> '
     f'(câu còn có thể là "... is the city of ..."), không trả lời câu hỏi. Sau "{prompts[1]}", "{rows[1][1]}" chiếm tới {rows[1][2]:.0%}.')

## 3. Loss của từng token

Khi train, input là chuỗi token, **label là chính chuỗi ấy dời đi một vị trí**: ở vị trí $t$, model phải đoán token $t+1$. Loss tại mỗi vị trí là $-\log P(\text{token đúng})$, tính bằng nat; loss của câu là trung bình. Tô màu mỗi token theo loss: càng đỏ càng bất ngờ với model.

In [ ]:
text = 'The Eiffel Tower is located in Paris, and it was completed in 1889 for the World Fair. My grandmother painted it purple yesterday.'
ids = tok(text, return_tensors='pt').input_ids.to(device)
with torch.no_grad(): out = lm(ids, labels=ids)
logp = out.logits[0, :-1].log_softmax(-1)
tok_loss = -logp[torch.arange(ids.shape[1] - 1), ids[0, 1:]].cpu().numpy()
check(f'trung bình loss từng token ({tok_loss.mean():.4f}) bằng loss model trả về ({out.loss.item():.4f})', abs(tok_loss.mean() - out.loss.item()) < 1e-3)

toks = [tok.decode(int(i)) for i in ids[0]]
mx = np.percentile(tok_loss, 95)
spans = [f'<span style="background:rgba(194,96,62,{min(l / mx, 1):.2f});padding:2px 1px;border-radius:3px" title="{l:.2f}">{html.escape(t)}</span>' for t, l in zip(toks[1:], tok_loss)]
display(HTML('<div style="font-size:17px;line-height:2;margin:8px 0"><span style="color:#999">' + html.escape(toks[0]) + '</span>' + ''.join(spans) + '</div>'))
top = np.argsort(-tok_loss)[:6]
table(pd.DataFrame({'token': [toks[i + 1] for i in top], 'đứng sau': [''.join(toks[max(0, i - 2):i + 1]) for i in top], 'loss (nat)': tok_loss[top], 'xác suất model cho': np.exp(-tok_loss[top])}),
      'Sáu token bất ngờ nhất', fmt={'loss (nat)': '{:.2f}', 'xác suất model cho': '{:.4f}'})
lt = {t.strip(): l for t, l in zip(toks[1:], tok_loss)}
note(f'Loss trung bình {tok_loss.mean():.2f} nat, tức perplexity {math.exp(tok_loss.mean()):.1f}. '
     f'Càng khó đoán càng tốn: "Tower" sau "Eiffel" chỉ {lt.get("Tower", float("nan")):.2f} nat, "Paris" {lt.get("Paris", float("nan")):.2f} nat, còn "purple" trong câu bịa ra tới {lt.get("purple", float("nan")):.2f} nat.')

## 4. Cách chọn token

Có phân phối rồi, mỗi bước chọn token thế nào?

- **greedy**: luôn lấy token xác suất cao nhất;
- **temperature** $T$: chia logits cho $T$ rồi lấy mẫu; $T < 1$ nhọn hơn, $T > 1$ bẹt hơn;
- **top-k**: chỉ lấy mẫu trong $k$ token cao nhất;
- **top-p**: chỉ lấy mẫu trong nhóm token cao nhất có tổng xác suất $\ge p$.

Sinh 40 token từ cùng một câu mở, đo **tỉ lệ lặp** (bigram trùng) và **log-xác suất trung bình** mỗi token theo chính GPT-2.

In [ ]:
start = 'In the middle of the night, the old lighthouse keeper heard'
ids0 = tok(start, return_tensors='pt').input_ids.to(device)
def gen(**kw):
    torch.manual_seed(0)
    with torch.no_grad():
        o = lm.generate(ids0, max_new_tokens=40, pad_token_id=tok.eos_token_id, **kw)
    new = o[0, ids0.shape[1]:]
    with torch.no_grad(): lp = lm(o).logits[0, ids0.shape[1] - 1:-1].log_softmax(-1)
    avg_lp = lp[torch.arange(len(new)), new].mean().item()
    w = tok.decode(new).split(); bg = list(zip(w, w[1:]))
    return tok.decode(new), 1 - len(set(bg)) / max(len(bg), 1), avg_lp
cfgs = [('greedy', dict(do_sample=False)), ('temperature 0.5', dict(do_sample=True, temperature=0.5, top_k=0)), ('temperature 1.0', dict(do_sample=True, temperature=1.0, top_k=0)),
        ('temperature 1.5', dict(do_sample=True, temperature=1.5, top_k=0)), ('top-k 40', dict(do_sample=True, top_k=40)), ('top-p 0.9', dict(do_sample=True, top_p=0.9, top_k=0))]
dec = pd.DataFrame([(n, *gen(**kw)) for n, kw in cfgs], columns=['cách chọn', 'văn bản sinh ra', 'tỉ lệ bigram lặp', 'log-xác suất trung bình'])
table(dec, f'Câu mở: "{start}"', fmt={'tỉ lệ bigram lặp': '{:.0%}', 'log-xác suất trung bình': '{:.2f}'})

with torch.no_grad(): lg = lm(ids0).logits[0, -1]
fig, ax = plt.subplots(figsize=(10, 3.4))
for T_, c in [(0.5, C['blue']), (1.0, C['gray']), (1.5, C['clay'])]:
    ax.plot(np.arange(1, 51), (lg / T_).softmax(-1).sort(descending=True).values[:50].cpu(), marker='o', ms=3, color=c, label=f'T = {T_}')
ax.set_yscale('log'); ax.set(xlabel='hạng của token', ylabel='xác suất (thang log)', title='Cùng logits, ba nhiệt độ: T nhỏ dồn xác suất về đầu'); ax.legend()
plt.tight_layout(); plt.show()
g = dec.iloc[0]
note(f"Greedy có log-xác suất cao nhất ({g['log-xác suất trung bình']:.2f}) nhưng dễ lặp ({g['tỉ lệ bigram lặp']:.0%} bigram lặp lại). T = 1.5 thì gần như không lặp nhưng log-xác suất tụt còn {dec.iloc[3, 3]:.2f}: văn bản bắt đầu vô nghĩa.")

## 5. Greedy không tối ưu cả chuỗi

Greedy chọn tốt nhất **từng bước**, không bảo đảm tốt nhất **cả chuỗi**: một bước kém hơn có thể mở ra những bước sau tốt hơn hẳn. **Beam search** giữ $B$ chuỗi tốt nhất ở mỗi bước. So tổng log-xác suất của 20 token sinh ra trên năm câu mở.

In [ ]:
starts = ['The best way to learn mathematics is', 'Scientists recently discovered that', 'My favourite food is', 'The football match ended with', 'In 2050, cities will']
rows = []
for s0 in starts:
    i0 = tok(s0, return_tensors='pt').input_ids.to(device)
    res = {}
    for n, kw in [('greedy', dict(do_sample=False)), ('beam 5', dict(num_beams=5, do_sample=False))]:
        with torch.no_grad():
            o = lm.generate(i0, max_new_tokens=20, pad_token_id=tok.eos_token_id, **kw)
            lp = lm(o).logits[0, i0.shape[1] - 1:-1].log_softmax(-1)
        res[n] = lp[torch.arange(o.shape[1] - i0.shape[1]), o[0, i0.shape[1]:]].sum().item()
    rows.append((s0, res['greedy'], res['beam 5']))
bm = pd.DataFrame(rows, columns=['câu mở', 'greedy: tổng log P', 'beam 5: tổng log P'])
bm['beam hơn'] = bm.iloc[:, 2] - bm.iloc[:, 1]
table(bm, '20 token sinh ra, log-xác suất cả chuỗi (càng gần 0 càng tốt)', fmt={c: '{:.2f}' for c in bm.columns[1:3]} | {'beam hơn': '{:+.2f}'})
check('beam search không thua greedy ở câu nào', (bm['beam hơn'] >= -1e-3).all())
note(f"Beam search tìm được chuỗi có tổng log-xác suất cao hơn greedy ở <b>{(bm['beam hơn'] > 1e-3).sum()}/{len(bm)}</b> câu, trung bình {bm['beam hơn'].mean():+.2f}.")

## 6. Perplexity của người và máy

Perplexity $= \exp(\text{loss trung bình})$: model "phân vân" giữa khoảng bấy nhiêu token mỗi bước. So ba loại văn bản cùng độ dài: đoạn **người viết** (WikiText-2 test), đoạn **GPT-2 tự sinh** bằng greedy, và chính đoạn người viết bị **xáo trộn thứ tự từ**.

In [ ]:
paras = [t for t in wiki['test']['text'] if len(t.split()) > 80 and not t.startswith(' =')][:20]
def ppl(text):
    ids = tok(text, return_tensors='pt').input_ids[:, :128].to(device)
    with torch.no_grad(): return math.exp(lm(ids, labels=ids).loss.item())
rng = np.random.default_rng(0); rows = []
for p_ in paras:
    w = p_.split(); human = ' '.join(w[:80]); shuffled = ' '.join(rng.permutation(w[:80]))
    pre = tok(' '.join(w[:15]), return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        o = lm.generate(pre, max_new_tokens=110, do_sample=False, repetition_penalty=1.3, pad_token_id=tok.eos_token_id)
    machine = tok.decode(o[0])
    rows.append((ppl(human), ppl(machine), ppl(shuffled)))
pp = pd.DataFrame(rows, columns=['người viết', 'GPT-2 sinh (greedy)', 'người viết, xáo trộn'])
table(pp.median().to_frame('perplexity trung vị').reset_index(names='văn bản'), f'{len(pp)} đoạn, mỗi đoạn tối đa 128 token', fmt={'perplexity trung vị': '{:.1f}'})
fig, ax = plt.subplots(figsize=(9, 3.4))
for k, (c, col) in enumerate(zip([C['blue'], C['clay'], C['gray']], pp.columns)):
    ax.scatter(pp[col], np.full(len(pp), k) + np.random.default_rng(k).uniform(-.12, .12, len(pp)), color=c, s=30)
ax.set_xscale('log'); ax.set_yticks(range(3), pp.columns); ax.set(xlabel='perplexity (thang log)', title='Mỗi chấm một đoạn'); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
check('văn bản máy sinh có perplexity thấp hơn văn bản người viết', pp.iloc[:, 1].median() < pp.iloc[:, 0].median())
note(f"Trung vị: người viết <b>{pp.iloc[:, 0].median():.0f}</b>, máy sinh <b>{pp.iloc[:, 1].median():.0f}</b>, xáo trộn {pp.iloc[:, 2].median():.0f}. "
     f"Văn bản máy sinh 'dễ đoán' với chính model sinh ra nó: perplexity thấp là một dấu hiệu (không phải bằng chứng) của văn bản do máy viết.")

## 7. Tóm tắt

In [ ]:
summary = [('từ hay đứng sau "the" nhất (bigram)', f"{after_the.iloc[0, 0]} ({after_the.iloc[0, 2]:.3f})"),
           (f'GPT-2: token cao nhất sau "{prompts[0]}"', f'{top_pred[1]} ({top_pred[2]:.2f})'),
           ('loss trung bình câu ở mục 3 / perplexity', f'{tok_loss.mean():.2f} / {math.exp(tok_loss.mean()):.1f}'),
           ('bigram lặp: greedy / top-p 0.9', f"{dec.iloc[0, 2]:.0%} / {dec.iloc[5, 2]:.0%}"),
           ('beam 5 hơn greedy (tổng log P, trung bình)', f"{bm['beam hơn'].mean():+.2f}"),
           ('perplexity trung vị: người / máy / xáo trộn', ' / '.join(f'{v:.0f}' for v in pp.median()))]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** viết câu của chính lớp ở mục 3 rồi xem token nào đỏ; đổi `repetition_penalty` ở mục 6 về 1.0 để thấy greedy lặp; thử câu tiếng Việt ở mục 2 và xem GPT-2 (học chủ yếu tiếng Anh) phân vân thế nào.